# Module 4: Complex Workflows (Autonomous Research Team) 🤖🔄🤖

In this final module, we move beyond simple routing. We will build a **Hierarchical Multi-Agent Team** that uses ADK's complex orchestration components:
1.  **SequentialAgent**: Executes agents in a predefined order (Chain of command).
2.  **LoopAgent**: Runs agents in a loop until a condition is met (Feedback loop).

### Architecture Overview

The architecture for this module involves a Sequential flow with an embedded Research Loop. You can find the Mermaid source for this diagram in `architecture_diagram.txt`.

### Our Use Case: The JPMorgan Research Memo
We will implement a system where:
-   A **Goal Refiner** clarifies the user's request.
-   An **Analyst** (RAG) and **Compliance Officer** (Evaluator) iterate in a loop until the report is professional.
-   A **Reporter** compiles the final findings into an Executive Memo.

## 1. Setup and Environment

In [ ]:
%pip install google-adk google-genai python-dotenv

In [ ]:
import os
import sys
from dotenv import load_dotenv
from google.adk.agents import Agent, SequentialAgent, LoopAgent
from google.genai import types
import uuid
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService

# Load API Key
load_dotenv('../.env.local')

# Add project root to path for specialist imports
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

## 2. Importing Specialist Agents
We reuse our RAG Analyst from Module 2.

In [ ]:
try:
    from module_02.rag_agent.agent import root_agent as rag_analyst
    print("✅ Specialist Analyst imported.")
except ImportError as e:
    print(f"❌ Could not find Analyst agent: {e}")

## 3. Defining the Research Team
We define the three layers of our workflow.

In [ ]:
# A. Goal Refiner
goal_refiner = Agent(
    model="gemini-2.5-flash",
    name="goal_refiner",
    instruction="""
    You are a Research Coordinator. Take the user's raw query and expand it into a detailed 
    technical goal for an investment analyst. Focus on JPMC policy and specific tickers if mentioned.
    """
)

# B. Compliance Officer (The Loop Critic)
compliance_officer = Agent(
    model="gemini-2.5-flash",
    name="compliance_officer",
    instruction="""
    Review the analyst's findings. If they are incomplete or not professional, give feedback.
    If the data is sufficient and high-quality, say: READY_FOR_SUMMARY.
    """
)

# C. Reporter (The Endpoint)
reporter = Agent(
    model="gemini-2.5-flash",
    name="reporter",
    instruction="""
    Take all researchers notes and compile them into a JPMorgan Executive Memo.
    Use professional headings and a 'Recommendation' section.
    """
)

print("✅ Orchestration components ready.")

## 4. Building the Composite Agents
Now we assemble the agents into **Groups**.

In [ ]:
# Define a callback for the LoopAgent
def completion_check(**kwargs):
    ctx = kwargs.get('callback_context')
    if ctx and ctx.user_content and ctx.user_content.parts:
        if "READY_FOR_SUMMARY" in ctx.user_content.parts[0].text:
            return True
    return False

# 1. The Autonomous Loop
research_loop = LoopAgent(
    name="research_loop",
    sub_agents=[rag_analyst, compliance_officer],
    max_iterations=3,
    after_agent_callback=completion_check
)

# 2. The Full Research Team Pipeline
research_team = SequentialAgent(
    name="research_team",
    sub_agents=[goal_refiner, research_loop, reporter]
)

print("🚀 Research Team is assembled.")

## 5. Execution
Let's see the autonomous team process a complex query.

In [ ]:
async def run_research(query: str):
    runner = Runner(agent=research_team, app_name="memo_factory")
    
    async for event in runner.run_async(
        user_id="notebook_user",
        session_id=str(uuid.uuid4()),
        new_message=types.Content(role="user", parts=[types.Part(text=query)])
    ):
        if event.content and event.content.parts:
            agent_name = event.agent_name if hasattr(event, 'agent_name') else "System"
            print(f"\n[{agent_name}] ------------------")
            print(event.content.parts[0].text)

await run_research("Analyze our strategy for Cloud/AI infra and ESG compliance.")